# Phase 04R: Teacher C → S3 Logit-KD Seed-2026

Confirmation run độc lập cho seed `2026`:

- Teacher C SigLIP ViT-B/16 **seed 2026**, frozen;
- Student S3 TinyViT-5M dùng initialization chung đã khóa ở Phase 02A;
- RNG, image order, caption rotation và stateless augmentation dùng seed `2026`;
- direct response/logit KD giữ nguyên `T=4`, `alpha=0.5`;
- MS COCO 2014 Karpathy Train+Restval;
- checkpoint selection chỉ bằng Validation CIDEr-D;
- Beam-3, length penalty 0.7;
- Held-out Test luôn `SEALED`.

## Điều kiện chạy

Phase 04Q seed 42 phải hoàn tất và cung cấp cả hai marker:

1. `PHASE4_TEACHER_C_S3_LOGIT_KD_SEED42_FULL_RUN_COMPLETE.json`;
2. `PHASE4_TEACHER_C_S3_LOGIT_KD_CONFIRMATION_SEEDS_AUTHORIZED.json`.

Notebook seed 123 và seed 2026 có thể chạy **song song trong hai Kaggle
sessions riêng**. Không thay đổi hyperparameter hoặc phương pháp KD.

## Kaggle inputs

1. Phase 01 formal output.
2. Phase 02A shared Student bundle.
3. Phase 02D S3 seed-2026 CE-only baseline output.
4. Phase 03K Teacher C seed-2026 output.
5. Phase 03L Teacher C aggregate/pool output.
6. Phase 04P training-readiness output.
7. Phase 04Q seed-42 completed output.
8. Offline SigLIP ViT-B/16.
9. MS COCO 2014 images.
10. Offline `pycocoevalcap==1.2` wheel.
11. Khi resume: output Phase 04R seed-2026 gần nhất.

GPU T4 hoặc mạnh hơn; Internet Off.


## Cell 1 — Authorization, matched seed và resume source

Xác minh Phase 04Q seed 42 đã hoàn tất, seed `2026` được phép chạy,
Teacher C seed `2026`, S3 CE-only baseline seed `2026` và optional
resume checkpoint. Chưa đọc dataset/checkpoint payload.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 2026
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_c_s3_logit_kd_seed2026"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
RESULT_DIR = OUTPUT_ROOT / "results"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, LOG_DIR, RESULT_DIR,
    VALIDATION_DIR, CHECKPOINT_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "Phase 04R bắt buộc GPU. Chọn Kaggle Settings -> "
        "Accelerator -> GPU T4 rồi chạy lại từ Cell 1."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE4P_COMPLETE = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_"
    "TRAINING_READINESS_COMPLETE.json"
)
FULL_RUN_AUTHORIZED = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_"
    "SEED42_FULL_RUN_AUTHORIZED.json"
)
CONFIRMATION_AUTHORIZED = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_"
    "CONFIRMATION_SEEDS_AUTHORIZED.json"
)
DEVELOPMENT_RUN_COMPLETE = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_"
    "SEED42_FULL_RUN_COMPLETE.json"
)
PHASE2_READY = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE1_READY = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
TEACHER_RUN_COMPLETE = exactly_one(
    "TEACHER_C_SEED2026_FULL_RUN_COMPLETE.json"
)
TEACHER_AGGREGATE_COMPLETE = exactly_one(
    "PHASE3_TEACHER_POOL_COMPLETE.json"
)
S3_BASELINE_COMPLETE = exactly_one(
    "S3_SEED2026_FULL_RUN_COMPLETE.json"
)

phase4p_complete = read_json(PHASE4P_COMPLETE)
full_run_authorized = read_json(FULL_RUN_AUTHORIZED)
confirmation_authorized = read_json(CONFIRMATION_AUTHORIZED)
development_run = read_json(DEVELOPMENT_RUN_COMPLETE)
phase2_ready = read_json(PHASE2_READY)
teacher_run = read_json(TEACHER_RUN_COMPLETE)
teacher_aggregate = read_json(TEACHER_AGGREGATE_COMPLETE)
s3_baseline = read_json(S3_BASELINE_COMPLETE)

assert phase4p_complete["status"] == "complete"
assert phase4p_complete["production_data_pipeline"] == "passed"
assert phase4p_complete["model_optimizer_engine"] == "passed"
assert phase4p_complete["validation_beam_engine"] == "passed"
assert phase4p_complete["discarded_pilot"] == "passed"
assert phase4p_complete["production_contract"] == "locked"
assert phase4p_complete["phase04q_full_run_authorized"] is True
assert full_run_authorized["status"] == "authorized"
assert full_run_authorized["experiment_id"] == "P4C_C_S3_LOGIT"
assert int(full_run_authorized["seed"]) == 42
assert confirmation_authorized["status"] == "authorized"
assert confirmation_authorized["experiment_id"] == "P4C_C_S3_LOGIT"
assert int(confirmation_authorized["development_seed_complete"]) == 42
assert 2026 in [
    int(value)
    for value in confirmation_authorized["confirmation_seeds"]
]
assert confirmation_authorized["method_changes_allowed"] is False
assert development_run["status"] == "complete"
assert int(development_run["seed"]) == 42
assert development_run["confirmation_seeds_authorized"] is True
assert s3_baseline["status"] == "complete"
assert s3_baseline["student_id"] == "S3"
assert int(s3_baseline["seed"]) == 2026

for marker in [
    phase4p_complete, full_run_authorized,
    confirmation_authorized, development_run,
    teacher_run, teacher_aggregate, s3_baseline,
]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

PHASE4P_ROOT = PHASE4P_COMPLETE.parent
production_ref = full_run_authorized["production_contract"]
PRODUCTION_CONTRACT_PATH = resolve_inside(
    PHASE4P_ROOT, production_ref["relative_path"]
)
assert sha256_file(PRODUCTION_CONTRACT_PATH) == (
    production_ref["sha256"]
)
PRODUCTION_CONTRACT = read_json(PRODUCTION_CONTRACT_PATH)
assert PRODUCTION_CONTRACT["status"] == "locked"
assert PRODUCTION_CONTRACT["kd"]["temperature"] == 4.0
assert PRODUCTION_CONTRACT["kd"]["alpha"] == 0.5

phase4p_manifest = resolve_inside(
    PHASE4P_ROOT,
    phase4p_complete["manifest"]["csv"]["relative_path"],
)
phase4p_sums = resolve_inside(
    PHASE4P_ROOT,
    phase4p_complete["manifest"]["sha256sums"]["relative_path"],
)
rows = list(csv.DictReader(
    phase4p_manifest.open("r", encoding="utf-8", newline="")
))
assert len(rows) == 11
expected_lines = []
for row in rows:
    artifact = resolve_inside(
        PHASE4P_ROOT, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]
    expected_lines.append(
        f'{row["sha256"]}  {row["relative_path"]}'
    )
actual_lines = [
    line.strip()
    for line in phase4p_sums.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]
assert actual_lines == expected_lines


PHASE4Q_ROOT = DEVELOPMENT_RUN_COMPLETE.parent
development_manifest = resolve_inside(
    PHASE4Q_ROOT,
    development_run["manifest"]["csv"]["relative_path"],
)
development_sums = resolve_inside(
    PHASE4Q_ROOT,
    development_run["manifest"]["sha256sums"]["relative_path"],
)
assert sha256_file(development_manifest) == (
    development_run["manifest"]["csv"]["sha256"]
)
assert sha256_file(development_sums) == (
    development_run["manifest"]["sha256sums"]["sha256"]
)
development_rows = list(csv.DictReader(
    development_manifest.open(
        "r", encoding="utf-8", newline=""
    )
))
assert len(development_rows) == 53
development_expected = []
for row in development_rows:
    artifact = resolve_inside(PHASE4Q_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]
    development_expected.append(
        f'{row["sha256"]}  {row["relative_path"]}'
    )
development_actual = [
    line.strip()
    for line in development_sums.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]
assert development_actual == development_expected

BASELINE_CIDER = float(
    s3_baseline["best_validation_metrics"]["CIDEr-D"]
)
BASELINE_BLEU4 = float(
    s3_baseline["best_validation_metrics"]["BLEU-4"]
)
BASELINE_CE = float(
    s3_baseline["best_validation_cross_entropy"]
)

resume_candidates = sorted(
    INPUT_ROOT.rglob(
        "teacher_c_s3_logit_kd_seed2026_last.pt"
    )
)
assert len(resume_candidates) <= 1, resume_candidates
INPUT_RESUME_CHECKPOINT = (
    resume_candidates[0].resolve()
    if resume_candidates else None
)


# Resolve the formal Teacher C/S3 model sources used by the locked Phase 04P
# engine. Checkpoint payloads are still not deserialized in Cell 1.
PHASE2_ROOT = PHASE2_READY.parent
phase2_manifest_path = resolve_inside(
    PHASE2_ROOT, phase2_ready["manifests"]["csv"]["relative_path"]
)
with phase2_manifest_path.open(
    "r", encoding="utf-8", newline=""
) as handle:
    phase2_rows = list(csv.DictReader(handle))
phase2_by_path = {row["relative_path"]: row for row in phase2_rows}
S3_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s3_encoder_checkpoint"],
)
DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
for checkpoint_path in [
    S3_INITIAL_CHECKPOINT, DECODER_INITIAL_CHECKPOINT,
]:
    relative = checkpoint_path.relative_to(PHASE2_ROOT).as_posix()
    assert relative in phase2_by_path
    assert sha256_file(checkpoint_path) == (
        phase2_by_path[relative]["sha256"]
    )
DECODER_CONTRACT = exactly_one(
    "shared_transformer_decoder_contract.json"
)
SHARED_DECODER_CONTRACT = DECODER_CONTRACT
SHARED_DECODER_INITIAL_CHECKPOINT = DECODER_INITIAL_CHECKPOINT

teacher_registry_path = resolve_inside(
    TEACHER_AGGREGATE_COMPLETE.parent,
    teacher_aggregate[
        "teacher_c_checkpoint_registry"
    ]["relative_path"],
)
assert sha256_file(teacher_registry_path) == (
    teacher_aggregate[
        "teacher_c_checkpoint_registry"
    ]["sha256"]
)
teacher_registry = read_json(teacher_registry_path)
teacher_seed2026_registry = [
    row for row in teacher_registry["entries"]
    if int(row["seed"]) == SEED
][0]
TEACHER_CHECKPOINT = exactly_one("teacher_c_seed2026_best.pt")
assert sha256_file(TEACHER_CHECKPOINT) == (
    teacher_seed2026_registry["best_checkpoint"]["sha256"]
)
TEACHER_C_BEST_CHECKPOINT = TEACHER_CHECKPOINT

siglip_sources = []
for source_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        source_manifest = read_json(source_path)
    except Exception:
        continue
    if source_manifest.get("repo_id") == (
        "google/siglip-base-patch16-224"
    ):
        siglip_sources.append(
            (source_path.resolve(), source_manifest)
        )
assert len(siglip_sources) == 1, siglip_sources
SIGLIP_SOURCE_MANIFEST_PATH, SIGLIP_SOURCE_MANIFEST = siglip_sources[0]
SIGLIP_SOURCE_DIR = SIGLIP_SOURCE_MANIFEST_PATH.parent
assert SIGLIP_SOURCE_MANIFEST["resolved_commit"] == (
    "7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed"
)
for source_row in SIGLIP_SOURCE_MANIFEST["files"]:
    source_artifact = resolve_inside(
        SIGLIP_SOURCE_DIR, source_row["relative_path"]
    )
    assert source_artifact.stat().st_size == int(
        source_row["size_bytes"]
    )
    assert sha256_file(source_artifact) == source_row["sha256"]

write_json(
    AUDIT_DIR / "phase4r_cell01_authorization.json",
    {
        "audit_version": "phase4r_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase4p_manifest_rows": len(rows),
        "phase4q_development_manifest_rows": len(development_rows),
        "development_seed42_run": "verified",
        "confirmation_seed": 2026,
        "confirmation_authorization": "verified",
        "method_changes_allowed": False,
        "production_contract_sha256": sha256_file(
            PRODUCTION_CONTRACT_PATH
        ),
        "s3_seed2026_baseline": {
            "cider_d": BASELINE_CIDER,
            "bleu_4": BASELINE_BLEU4,
            "validation_ce": BASELINE_CE,
        },
        "resume_checkpoint_found": (
            INPUT_RESUME_CHECKPOINT is not None
        ),
        "dataset_read": False,
        "checkpoint_payload_read": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04R CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 04P manifest        : 11 / 11 VERIFIED")
print("Seed42 development run    : VERIFIED")
print("Confirmation authorization: VERIFIED")
print("Method changes allowed    : False")
print("Production contract       : LOCKED")
print("S3 seed2026 baseline CIDEr-D:", BASELINE_CIDER)
print("Resume checkpoint found   :", INPUT_RESUME_CHECKPOINT is not None)
print("GPU                       :", torch.cuda.get_device_name(0))
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : rebuild production data pipeline")


PHASE 04R CELL 1 STATUS
Status                    : PASSED
Phase 04P manifest        : 11 / 11 VERIFIED
Seed42 development run    : VERIFIED
Confirmation authorization: VERIFIED
Method changes allowed    : False
Production contract       : LOCKED
S3 seed2026 baseline CIDEr-D: 1.070901867446434
Resume checkpoint found   : False
GPU                       : Tesla T4
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : rebuild production data pipeline


## Cell 2 — Rebuild production data pipeline

Khôi phục đúng Phase 04P image-balanced sampler, stateless shared geometry,
SigLIP/ImageNet dual normalization, Validation captions/images và resume cursor.


In [2]:
CELL_STARTED_AT = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF


TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")


def find_image_leaf(name, prefix):
    matches = []
    for path in INPUT_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
    filters=[
        ("experiment_split", "in", ["train", "validation"])
    ],
)
TRAIN_PATHS = {}
VALIDATION_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    path = leaf / Path(row["image_path"]).name
    target = (
        TRAIN_PATHS
        if row["experiment_split"] == "train"
        else VALIDATION_PATHS
    )
    target[image_id] = path
assert len(TRAIN_PATHS) == 113287
assert len(VALIDATION_PATHS) == 5000

TRAIN_CAPTIONS = defaultdict(list)
for batch_arrow in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in batch_arrow.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append(
            (
                int(row["annotation_id"]),
                tuple(int(value) for value in row["token_ids"]),
            )
        )
assert len(TRAIN_CAPTIONS) == 113287
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for batch_arrow in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in batch_arrow.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append(
            (
                int(row["annotation_id"]),
                image_id,
                tuple(int(value) for value in row["token_ids"]),
            )
        )
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000

TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))
SIGLIP_MEAN = torch.tensor(
    (0.5, 0.5, 0.5)
).view(3, 1, 1)
SIGLIP_STD = torch.tensor(
    (0.5, 0.5, 0.5)
).view(3, 1, 1)
IMAGENET_MEAN = torch.tensor(
    (0.485, 0.456, 0.406)
).view(3, 1, 1)
IMAGENET_STD = torch.tensor(
    (0.229, 0.224, 0.225)
).view(3, 1, 1)


def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|phase4r"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8],
        "big",
    ) % (2 ** 63 - 1)


def shared_train_geometry(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = (
            transforms.RandomResizedCrop.get_params(
                image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
            )
        )
        do_flip = bool(torch.rand(()) < 0.5)
    image = TF.resized_crop(
        image, top, left, height, width,
        size=[224, 224],
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    if do_flip:
        image = TF.hflip(image)
    return TF.to_tensor(image)


def validation_geometry(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    image = TF.center_crop(image, [224, 224])
    return TF.to_tensor(image)


def dual_normalize(base):
    return (
        (base - SIGLIP_MEAN) / SIGLIP_STD,
        (base - IMAGENET_MEAN) / IMAGENET_STD,
    )


class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, token_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        with Image.open(TRAIN_PATHS[image_id]) as image:
            base = shared_train_geometry(
                image.convert("RGB"), self.epoch, image_id
            )
        teacher_image, student_image = dual_normalize(base)
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "teacher_image": teacher_image,
            "student_image": student_image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as image:
            base = validation_geometry(image.convert("RGB"))
        _, student_image = dual_normalize(base)
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": student_image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as image:
            base = validation_geometry(image.convert("RGB"))
        _, student_image = dual_normalize(base)
        return {"image_id": image_id, "student_image": student_image}


class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(
            self.length, generator=generator
        ).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor


def caption_collate(rows):
    maximum = max(row["token_ids"].numel() for row in rows)
    full = torch.zeros(len(rows), maximum, dtype=torch.long)
    for index, row in enumerate(rows):
        full[index, :row["token_ids"].numel()] = row["token_ids"]
    result = {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
    }
    result["decoder_attention_mask"] = (
        result["decoder_input_ids"].ne(0)
    )
    if "teacher_image" in rows[0]:
        result["teacher_images"] = torch.stack(
            [row["teacher_image"] for row in rows]
        )
    return result


def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }


TRAIN_DATASET = ProductionTrainDataset()
VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()


def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    sampler = EpochCursorSampler(
        len(TRAIN_DATASET), epoch=epoch, cursor=cursor
    )
    return DataLoader(
        TRAIN_DATASET,
        batch_size=32,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=caption_collate,
    )


VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

loader_a = build_train_loader(0)
loader_b = build_train_loader(0)
batch_a = next(iter(loader_a))
batch_b = next(iter(loader_b))
for key in [
    "image_ids", "annotation_ids", "teacher_images",
    "student_images", "decoder_input_ids", "labels",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert len(set(batch_a["image_ids"].tolist())) == 32

write_json(
    AUDIT_DIR / "phase4r_cell02_data_pipeline.json",
    {
        "audit_version": "phase4r_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "train_images_per_epoch": len(TRAIN_DATASET),
        "captions_per_image_per_epoch": 1,
        "validation_captions": len(VALIDATION_CAPTION_DATASET),
        "validation_images": len(VALIDATION_IMAGE_DATASET),
        "physical_batch_size": 32,
        "shared_train_geometry": True,
        "stateless_augmentation": True,
        "two_loader_exact": True,
        "resume_cursor_supported": True,
        "validation_caption_text_read": True,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Train images / epoch      :", len(TRAIN_DATASET))
print("Captions / image / epoch  : 1")
print("Validation captions       :", len(VALIDATION_CAPTION_DATASET))
print("Validation images         :", len(VALIDATION_IMAGE_DATASET))
print("Physical/effective batch  : 32 / 64")
print("Shared train geometry     : True")
print("Stateless augmentation    : True")
print("Two-loader exact          : True")
print("Resume cursor supported   : True")
print("Test Set status           : SEALED")
print("Next                      : model and production optimizer")


PHASE 04P CELL 2 STATUS
Status                    : PASSED
Train images / epoch      : 113287
Captions / image / epoch  : 1
Validation captions       : 25010
Validation images         : 5000
Physical/effective batch  : 32 / 64
Shared train geometry     : True
Stateless augmentation    : True
Two-loader exact          : True
Resume cursor supported   : True
Test Set status           : SEALED
Next                      : model and production optimizer


## Cell 3 — Rebuild models, optimizer và KD step engine

Strict-load Teacher C/S3, đóng băng Teacher, dựng AdamW/warmup-cosine/
GradScaler cùng token-balanced Direct Logit-KD engine.


In [3]:
CELL_STARTED_AT = time.perf_counter()
import timm
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from transformers import SiglipModel


decoder_config = read_json(
    SHARED_DECODER_CONTRACT
)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080
assert int(decoder_config["pad_id"]) == 0
assert int(decoder_config["maximum_decoder_steps"]) == 31


class SharedCaptionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(
            config["maximum_decoder_steps"]
        )
        self.token_embedding = nn.Embedding(
            self.vocabulary_size,
            self.d_model,
            padding_idx=self.pad_id,
        )
        self.position_embedding = nn.Embedding(
            self.maximum_decoder_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(
            float(config["dropout"])
        )
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation=config["activation"],
            batch_first=True,
            norm_first=True,
            bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(
            torch.zeros(self.vocabulary_size)
        )

    def forward(
        self,
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask=None,
    ):
        batch, length = decoder_input_ids.shape
        positions = torch.arange(
            length, device=decoder_input_ids.device
        ).unsqueeze(0).expand(batch, -1)
        target = (
            self.token_embedding(decoder_input_ids)
            * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=decoder_input_ids.device,
            ),
            diagonal=1,
        )
        padding = (
            None
            if decoder_attention_mask is None
            else ~decoder_attention_mask.bool()
        )
        hidden = self.transformer_decoder(
            tgt=target,
            memory=visual_memory,
            tgt_mask=causal,
            tgt_key_padding_mask=padding,
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden,
            self.token_embedding.weight,
            self.output_bias,
        )


class TeacherVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, 196, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, patches):
        assert patches.shape[1:] == (196, 768)
        return self.output_norm(
            self.projection(patches) + self.position_embedding
        )


def select_tensor_from_feature_output(feature_output):
    if torch.is_tensor(feature_output):
        return feature_output
    if isinstance(feature_output, dict):
        tensors = [
            value for value in feature_output.values()
            if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    if isinstance(feature_output, (tuple, list)):
        tensors = [
            value for value in feature_output
            if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    raise TypeError(type(feature_output).__name__)


class StudentVisualEncoderAdapter(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.projection = nn.Linear(320, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = select_tensor_from_feature_output(
            self.backbone.forward_features(images)
        )
        assert tuple(feature.shape[1:]) == (320, 7, 7)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 320)
        return self.output_norm(
            self.projection(tokens)
            + self.visual_position_embedding
        )


os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
full_siglip = SiglipModel.from_pretrained(
    SIGLIP_SOURCE_DIR, local_files_only=True
)
TEACHER_ENCODER = full_siglip.vision_model
del full_siglip
TEACHER_ADAPTER = TeacherVisualAdapter()
TEACHER_DECODER = SharedCaptionDecoder(decoder_config)

tinyvit_backbone = timm.create_model(
    "tiny_vit_5m_224.dist_in22k_ft_in1k",
    pretrained=False,
)
if hasattr(tinyvit_backbone, "reset_classifier"):
    tinyvit_backbone.reset_classifier(
        num_classes=0, global_pool=""
    )
removed_heads = []
for head_name in ["head", "head_dist", "classifier", "fc"]:
    module = getattr(tinyvit_backbone, head_name, None)
    if (
        isinstance(module, nn.Module)
        and not isinstance(module, nn.Identity)
    ):
        setattr(tinyvit_backbone, head_name, nn.Identity())
        removed_heads.append(head_name)
assert removed_heads == ["head"]
STUDENT_ENCODER = StudentVisualEncoderAdapter(tinyvit_backbone)
STUDENT_DECODER = SharedCaptionDecoder(decoder_config)

with torch.serialization.safe_globals([TorchVersion]):
    teacher_checkpoint = torch.load(
        TEACHER_C_BEST_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
    student_encoder_checkpoint = torch.load(
        S3_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
    student_decoder_checkpoint = torch.load(
        SHARED_DECODER_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )

assert teacher_checkpoint["schema_version"] == (
    "phase3_teacher_c_seed2026_checkpoint_v1"
)
assert teacher_checkpoint["teacher_id"] == "C"
assert int(teacher_checkpoint["seed"]) == SEED
assert teacher_checkpoint["test_set_status"] == "sealed"
assert teacher_checkpoint["test_set_used"] is False

teacher_encoder_state = TEACHER_ENCODER.state_dict()
unknown_teacher_keys = set(
    teacher_checkpoint["encoder_candidate_state"]
) - set(teacher_encoder_state)
assert not unknown_teacher_keys, unknown_teacher_keys
teacher_encoder_state.update(
    teacher_checkpoint["encoder_candidate_state"]
)
TEACHER_ENCODER.load_state_dict(
    teacher_encoder_state, strict=True
)
teacher_adapter_load = TEACHER_ADAPTER.load_state_dict(
    teacher_checkpoint["adapter_state"], strict=True
)
teacher_decoder_load = TEACHER_DECODER.load_state_dict(
    teacher_checkpoint["decoder_state"], strict=True
)
student_encoder_load = STUDENT_ENCODER.load_state_dict(
    student_encoder_checkpoint["state_dict"], strict=True
)
student_decoder_load = STUDENT_DECODER.load_state_dict(
    student_decoder_checkpoint["state_dict"], strict=True
)
for result in [
    teacher_adapter_load,
    teacher_decoder_load,
    student_encoder_load,
    student_decoder_load,
]:
    assert not result.missing_keys
    assert not result.unexpected_keys

STUDENT_ENCODER_INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
STUDENT_DECODER_INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}

del teacher_checkpoint
del student_encoder_checkpoint
del student_decoder_checkpoint
del teacher_encoder_state

TEACHER_ENCODER = TEACHER_ENCODER.to(device)
TEACHER_ADAPTER = TEACHER_ADAPTER.to(device)
TEACHER_DECODER = TEACHER_DECODER.to(device)
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

for module in [
    TEACHER_ENCODER,
    TEACHER_ADAPTER,
    TEACHER_DECODER,
]:
    module.eval()
    for parameter in module.parameters():
        parameter.requires_grad_(False)


def teacher_forward(images, decoder_ids, decoder_mask):
    patches = TEACHER_ENCODER(
        pixel_values=images
    ).last_hidden_state
    assert patches.shape[1:] == (196, 768)
    memory = TEACHER_ADAPTER(patches)
    return TEACHER_DECODER(
        decoder_ids, memory, decoder_mask
    )


def student_forward(images, decoder_ids, decoder_mask):
    memory = STUDENT_ENCODER(images)
    assert memory.shape[1:] == (49, 256)
    return STUDENT_DECODER(
        decoder_ids, memory, decoder_mask
    )


def set_student_train_mode():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    frozen_batchnorm = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(
            module, nn.modules.batchnorm._BatchNorm
        ):
            module.eval()
            frozen_batchnorm += 1
    assert frozen_batchnorm == 27
    return frozen_batchnorm


teacher_parameter_count = sum(
    parameter.numel()
    for module in [
        TEACHER_ENCODER,
        TEACHER_ADAPTER,
        TEACHER_DECODER,
    ]
    for parameter in module.parameters()
)
student_parameter_count = sum(
    parameter.numel()
    for module in [STUDENT_ENCODER, STUDENT_DECODER]
    for parameter in module.parameters()
)
assert student_parameter_count == 10_925_684
assert all(
    not parameter.requires_grad
    for module in [
        TEACHER_ENCODER,
        TEACHER_ADAPTER,
        TEACHER_DECODER,
    ]
    for parameter in module.parameters()
)


STUDENT_ENCODER_INITIAL = STUDENT_ENCODER_INITIAL_STATE
STUDENT_DECODER_INITIAL = STUDENT_DECODER_INITIAL_STATE


def set_student_train():
    return set_student_train_mode()


def teacher_logits(batch):
    return teacher_forward(
        batch["teacher_images"],
        batch["decoder_input_ids"],
        batch["decoder_attention_mask"],
    ).detach()


def student_logits(batch):
    return student_forward(
        batch["student_images"],
        batch["decoder_input_ids"],
        batch["decoder_attention_mask"],
    )

TEMPERATURE = 4.0
ALPHA = 0.5
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 2
STEPS_PER_EPOCH = math.ceil(113287 / 64)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())


def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4,
    )

    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / WARMUP_STEPS
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def loss_sums(student, teacher, labels):
    valid = labels.ne(0)
    valid_count = int(valid.sum())
    ce_sum = F.cross_entropy(
        student.float().reshape(-1, 10080),
        labels.reshape(-1),
        ignore_index=0,
        reduction="sum",
    )
    student_log_probs = F.log_softmax(
        student.float() / TEMPERATURE, dim=-1
    )
    teacher_probs = F.softmax(
        teacher.float() / TEMPERATURE, dim=-1
    )
    kl_sum = (
        F.kl_div(
            student_log_probs, teacher_probs, reduction="none"
        ).sum(dim=-1)
        * valid
    ).sum() * (TEMPERATURE ** 2)
    combined_sum = (1 - ALPHA) * ce_sum + ALPHA * kl_sum
    return combined_sum, ce_sum, kl_sum, valid_count


def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }


def optimizer_step(
    microbatches, optimizer, scheduler, scaler,
    retry_limit=10,
):
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        combined_total = 0.0
        ce_total = 0.0
        kd_total = 0.0
        valid_total = 0
        for cpu_batch in microbatches:
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                teacher = teacher_logits(current)
                student = student_logits(current)
                combined, ce_sum, kd_sum, valid = loss_sums(
                    student, teacher, current["labels"]
                )
            scaler.scale(
                combined / BACKWARD_DIVISOR
            ).backward()
            combined_total += float(combined.detach())
            ce_total += float(ce_sum.detach())
            kd_total += float(kd_sum.detach())
            valid_total += valid
        scaler.unscale_(optimizer)
        correction = BACKWARD_DIVISOR / valid_total
        for parameter in (
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters())
        ):
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters()),
            1.0,
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        assert torch.isfinite(gradient_norm)
        scheduler.step()
        return {
            "combined_loss": combined_total / valid_total,
            "ce": ce_total / valid_total,
            "kd_kl_t2": kd_total / valid_total,
            "valid_tokens": valid_total,
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }


optimizer, scheduler, scaler = build_optimizer_state()
assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771

write_json(
    CONFIG_DIR / "phase4r_production_optimizer_contract.json",
    {
        "contract_version": "phase4r_optimizer_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "optimizer": "AdamW",
        "learning_rates": {
            "backbone": 1e-4, "adapter": 3e-4, "decoder": 3e-4,
        },
        "weight_decay": 1e-4,
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "maximum_epochs": MAX_EPOCHS,
        "total_steps": TOTAL_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "minimum_lr_factor": 0.05,
        "mixed_precision": "FP16 plus GradScaler",
        "gradscaler_initial_scale": 1024.0,
        "overflow_retry_limit": 10,
        "backward_divisor": BACKWARD_DIVISOR,
        "gradient_clip": 1.0,
        "teacher_frozen": True,
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase4r_cell03_model_optimizer.json",
    {
        "audit_version": "phase4r_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "strict_loads": 5,
        "teacher_frozen": True,
        "student_parameters": 10925684,
        "optimizer_parameter_groups": 3,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "total_steps": TOTAL_STEPS,
        "scheduler_advanced": False,
        "optimizer_step_performed": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04P CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Strict checkpoint loads   : 5 / 5 PASSED")
print("Teacher frozen            : True")
print("Student parameters        : 10925684")
print("Optimizer groups          : 3 / 3")
print("Physical/effective batch  : 32 / 64")
print("Steps / epoch             :", STEPS_PER_EPOCH)
print("Total optimizer steps     :", TOTAL_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("FP16 / GradScaler         : ENABLED / 1024")
print("Overflow retry limit      : 10")
print("Token-balanced accumulation: READY")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Next                      : Validation and Beam-3 engine")


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

PHASE 04P CELL 3 STATUS
Status                    : PASSED
Strict checkpoint loads   : 5 / 5 PASSED
Teacher frozen            : True
Student parameters        : 10925684
Optimizer groups          : 3 / 3
Physical/effective batch  : 32 / 64
Steps / epoch             : 1771
Total optimizer steps     : 35420
Warmup steps              : 1771
FP16 / GradScaler         : ENABLED / 1024
Overflow retry limit      : 10
Token-balanced accumulation: READY
Optimizer step performed  : False
Test Set status           : SEALED
Next                      : Validation and Beam-3 engine


## Cell 4 — Rebuild full Validation và batched Beam-3

Khóa teacher-forced CE, Beam-3 trên 5.000 ảnh, COCO metrics và deterministic
selection order CIDEr-D/BLEU-4/CE/earlier epoch.


In [4]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary_payload = read_json(VOCAB_PATH)
if "id_to_token" in vocabulary_payload:
    raw_id_to_token = vocabulary_payload["id_to_token"]
    if isinstance(raw_id_to_token, list):
        ID_TO_TOKEN = {
            index: token
            for index, token in enumerate(raw_id_to_token)
        }
    else:
        ID_TO_TOKEN = {
            int(key): value
            for key, value in raw_id_to_token.items()
        }
else:
    token_to_id = vocabulary_payload["token_to_id"]
    ID_TO_TOKEN = {int(value): key for key, value in token_to_id.items()}


def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)


def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(
        tokenized_gt, tokenized_pred
    )
    meteor, _ = Meteor().compute_score(
        tokenized_gt, tokenized_pred
    )
    rouge, _ = Rouge().compute_score(
        tokenized_gt, tokenized_pred
    )
    cider, _ = Cider().compute_score(
        tokenized_gt, tokenized_pred
    )
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }


smoke_ids = list(VALIDATION_IMAGE_IDS[:32])
smoke_references = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in smoke_ids
}
smoke_hypotheses = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in smoke_ids
}
metric_smoke = compute_coco_metrics(
    smoke_references, smoke_hypotheses
)
assert all(math.isfinite(value) for value in metric_smoke.values())


def teacher_forced_ce(loader, maximum_batches=None):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch_index, cpu_batch in enumerate(loader):
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                logits = student_logits(current)
            valid = current["labels"].ne(0)
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                current["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
            total_loss += float(loss_sum)
            total_tokens += int(valid.sum())
            if (
                maximum_batches is not None
                and batch_index + 1 >= maximum_batches
            ):
                break
    return total_loss / total_tokens, total_tokens


def beam_generate_one(student_image, beam_size=3):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    with torch.no_grad():
        memory = STUDENT_ENCODER(
            student_image.unsqueeze(0).to(device)
        )
        beams = [([1], 0.0, False)]
        for step in range(31):
            candidates = []
            active = [item for item in beams if not item[2]]
            if not active:
                break
            for tokens, score, finished in beams:
                if finished:
                    candidates.append((tokens, score, True))
                    continue
                ids = torch.tensor(
                    tokens, dtype=torch.long, device=device
                ).unsqueeze(0)
                mask = ids.ne(0)
                logits = STUDENT_DECODER(ids, memory, mask)[:, -1]
                log_probs = F.log_softmax(logits.float(), dim=-1)[0]
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, beam_size)
                for value, index in zip(
                    values.tolist(), indices.tolist()
                ):
                    next_tokens = tokens + [index]
                    candidates.append(
                        (
                            next_tokens,
                            score + value,
                            index == 2,
                        )
                    )
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(
                candidates, key=rank, reverse=True
            )[:beam_size]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
        return best[0], best[1]


validation_smoke_batch = next(iter(VALIDATION_IMAGE_LOADER))
tokens_a, score_a = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
tokens_b, score_b = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
assert tokens_a == tokens_b
assert score_a == score_b
smoke_ce, smoke_tokens = teacher_forced_ce(
    VALIDATION_CAPTION_LOADER, maximum_batches=1
)
assert math.isfinite(smoke_ce)

REFERENCES_PATH = (
    VALIDATION_DIR / "karpathy_validation_references.json"
)
write_json(
    REFERENCES_PATH,
    {
        str(image_id): captions
        for image_id, captions in VALIDATION_REFERENCES.items()
    },
)
write_json(
    AUDIT_DIR / "phase4r_cell04_validation_engine.json",
    {
        "audit_version": "phase4r_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "metric_backend": "pycocoevalcap 1.2",
        "metric_smoke": metric_smoke,
        "teacher_forced_smoke_ce": smoke_ce,
        "teacher_forced_smoke_tokens": smoke_tokens,
        "beam_size": 3,
        "length_penalty": 0.7,
        "beam_repeat_exact": True,
        "full_validation_run": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04R CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("COCO metrics              : 5 / 5 WIRED")
print("Teacher-forced CE smoke   :", smoke_ce)
print("Beam size / penalty       : 3 / 0.7")
print("Beam repeat exact         : True")
print("Full Validation run       : False")
print("Test Set status           : SEALED")
print("Next                      : full training or exact resume")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 1986 tokens at 23191.91 tokens per second.
PTBTokenizer tokenized 385 tokens at 5469.54 tokens per second.


{'testlen': 328, 'reflen': 328, 'guess': [328, 296, 264, 232], 'correct': [328, 296, 264, 232]}
ratio: 0.9999999999969512
PHASE 04R CELL 4 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
COCO metrics              : 5 / 5 WIRED
Teacher-forced CE smoke   : 11.347428206559066
Beam size / penalty       : 3 / 0.7
Beam repeat exact         : True
Full Validation run       : False
Test Set status           : SEALED
Next                      : full training or exact resume


## Cell 5 — Full training hoặc exact resume

Huấn luyện tối đa 20 epoch, atomic last mỗi 250 optimizer step, Validation
sau mỗi epoch, early stopping và resume đúng optimizer-step boundary.


In [5]:
CELL_STARTED_AT = time.perf_counter()
from torch.torch_version import TorchVersion


LAST_CHECKPOINT = (
    CHECKPOINT_DIR / "teacher_c_s3_logit_kd_seed2026_last.pt"
)
BEST_CHECKPOINT = (
    CHECKPOINT_DIR / "teacher_c_s3_logit_kd_seed2026_best.pt"
)
STEP_LOG = LOG_DIR / "teacher_c_s3_logit_kd_seed2026_steps.jsonl"
EPOCH_HISTORY_CSV = (
    LOG_DIR / "teacher_c_s3_logit_kd_seed2026_epoch_history.csv"
)
RUN_STATE_PATH = OUTPUT_ROOT / "PHASE4R_SEED2026_RUN_STATE.json"

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3
VOCABULARY_SIZE = 10080
BEAM_SIZE = 3
LENGTH_PENALTY = 0.7
MAXIMUM_DECODER_STEPS = 31
MINIMUM_LEXICAL_TOKENS = 2
FORBIDDEN_GENERATION_TOKEN_IDS = (PAD_ID, BOS_ID, UNK_ID)


def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)


def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_checkpoint(
    path, optimizer, scheduler, scaler, progress,
    history, best_result, no_improve_epochs,
):
    atomic_save(
        path,
        {
            "schema_version": "phase4_teacher_c_s3_kd_checkpoint_v1",
            "experiment_id": "P4C_C_S3_LOGIT",
            "seed": SEED,
            "student_encoder_state": clone_cpu(
                STUDENT_ENCODER.state_dict()
            ),
            "student_decoder_state": clone_cpu(
                STUDENT_DECODER.state_dict()
            ),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "progress": progress,
            "history": history,
            "best_result": best_result,
            "no_improve_epochs": no_improve_epochs,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "teacher_checkpoint_sha256": sha256_file(
                TEACHER_CHECKPOINT
            ),
            "production_contract_sha256": sha256_file(
                PRODUCTION_CONTRACT_PATH
            ),
            "temperature": TEMPERATURE,
            "alpha": ALPHA,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )


def load_checkpoint(path, optimizer, scheduler, scaler):
    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
    assert checkpoint["schema_version"] == (
        "phase4_teacher_c_s3_kd_checkpoint_v1"
    )
    assert checkpoint["experiment_id"] == "P4C_C_S3_LOGIT"
    assert int(checkpoint["seed"]) == 2026
    assert checkpoint["test_set_status"] == "sealed"
    assert checkpoint["test_set_used"] is False
    assert checkpoint["teacher_checkpoint_sha256"] == (
        sha256_file(TEACHER_CHECKPOINT)
    )
    assert checkpoint["production_contract_sha256"] == (
        sha256_file(PRODUCTION_CONTRACT_PATH)
    )
    STUDENT_ENCODER.load_state_dict(
        checkpoint["student_encoder_state"], strict=True
    )
    STUDENT_DECODER.load_state_dict(
        checkpoint["student_decoder_state"], strict=True
    )
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])
    torch.set_rng_state(checkpoint["cpu_rng"])
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
    return (
        checkpoint["progress"],
        checkpoint["history"],
        checkpoint["best_result"],
        int(checkpoint["no_improve_epochs"]),
    )


@torch.inference_mode()
def batched_beam_search(visual_memory):
    batch_size, memory_length, memory_dimension = visual_memory.shape
    assert memory_length == 49 and memory_dimension == 256
    beam_tokens = torch.full(
        (batch_size, BEAM_SIZE, 1),
        BOS_ID, dtype=torch.long, device=device,
    )
    beam_scores = torch.full(
        (batch_size, BEAM_SIZE),
        float("-inf"), dtype=torch.float32, device=device,
    )
    beam_scores[:, 0] = 0.0
    beam_finished = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.bool, device=device
    )
    beam_lengths = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.long, device=device
    )
    expanded_memory = (
        visual_memory.unsqueeze(1)
        .expand(-1, BEAM_SIZE, -1, -1)
        .reshape(batch_size * BEAM_SIZE, memory_length, memory_dimension)
        .contiguous()
    )
    for generation_step in range(MAXIMUM_DECODER_STEPS):
        flat_tokens = beam_tokens.reshape(
            batch_size * BEAM_SIZE, -1
        )
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = STUDENT_DECODER(
                flat_tokens,
                expanded_memory,
                flat_tokens.ne(PAD_ID),
            )
        log_probs = F.log_softmax(
            logits[:, -1, :].float(), dim=-1
        ).reshape(batch_size, BEAM_SIZE, VOCABULARY_SIZE)
        log_probs[:, :, list(FORBIDDEN_GENERATION_TOKEN_IDS)] = (
            float("-inf")
        )
        if generation_step < MINIMUM_LEXICAL_TOKENS:
            log_probs[:, :, EOS_ID] = float("-inf")
        if beam_finished.any():
            log_probs = log_probs.masked_fill(
                beam_finished.unsqueeze(-1), float("-inf")
            )
            finished_indices = beam_finished.nonzero(as_tuple=False)
            log_probs[
                finished_indices[:, 0],
                finished_indices[:, 1],
                EOS_ID,
            ] = 0.0
        candidate_scores = beam_scores.unsqueeze(-1) + log_probs
        next_scores, flat_indices = torch.topk(
            candidate_scores.reshape(
                batch_size, BEAM_SIZE * VOCABULARY_SIZE
            ),
            k=BEAM_SIZE,
            dim=-1,
        )
        parent = flat_indices // VOCABULARY_SIZE
        next_tokens = flat_indices % VOCABULARY_SIZE
        parent_tokens = torch.gather(
            beam_tokens,
            1,
            parent.unsqueeze(-1).expand(
                -1, -1, beam_tokens.shape[-1]
            ),
        )
        parent_finished = torch.gather(
            beam_finished, 1, parent
        )
        parent_lengths = torch.gather(
            beam_lengths, 1, parent
        )
        beam_tokens = torch.cat(
            [parent_tokens, next_tokens.unsqueeze(-1)], dim=-1
        )
        beam_scores = next_scores
        beam_lengths = parent_lengths + (~parent_finished).long()
        beam_finished = parent_finished | (next_tokens == EOS_ID)
        if bool(beam_finished.all()):
            break
    normalized = beam_scores / (
        beam_lengths.clamp_min(1).float() ** LENGTH_PENALTY
    )
    best = normalized.argmax(dim=1)
    batch_indices = torch.arange(batch_size, device=device)
    return {
        "token_ids": beam_tokens[batch_indices, best].cpu(),
        "raw_scores": beam_scores[batch_indices, best].cpu(),
        "normalized_scores": normalized[batch_indices, best].cpu(),
    }


@torch.inference_mode()
def generate_validation_captions():
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    hypotheses = {}
    records = {}
    started = time.perf_counter()
    for batch_index, cpu_batch in enumerate(
        VALIDATION_IMAGE_LOADER
    ):
        image_ids = [
            int(value) for value in cpu_batch["image_ids"].tolist()
        ]
        images = cpu_batch["student_images"].to(
            device, non_blocking=True
        )
        with torch.amp.autocast("cuda", dtype=torch.float16):
            memory = STUDENT_ENCODER(images)
        beam = batched_beam_search(memory)
        for row_index, image_id in enumerate(image_ids):
            token_ids = [
                int(value)
                for value in beam["token_ids"][row_index].tolist()
            ]
            caption = decode_tokens(token_ids)
            assert caption
            assert image_id not in hypotheses
            hypotheses[image_id] = caption
            records[image_id] = {
                "caption": caption,
                "token_ids": token_ids,
                "raw_score": float(
                    beam["raw_scores"][row_index]
                ),
                "normalized_score": float(
                    beam["normalized_scores"][row_index]
                ),
            }
        if (batch_index + 1) % 25 == 0:
            print(
                f"Generated {len(hypotheses):,}/5,000 "
                "Validation captions"
            )
    assert len(hypotheses) == 5000
    assert set(hypotheses) == set(VALIDATION_REFERENCES)
    return hypotheses, records, time.perf_counter() - started


def evaluate_validation(epoch):
    ce, valid_tokens = teacher_forced_ce(
        VALIDATION_CAPTION_LOADER
    )
    hypotheses, records, generation_seconds = (
        generate_validation_captions()
    )
    metrics = compute_coco_metrics(
        VALIDATION_REFERENCES, hypotheses
    )
    result = {
        "epoch": int(epoch),
        "teacher_forced_ce": float(ce),
        "valid_tokens": int(valid_tokens),
        "metrics": metrics,
        "prediction_count": len(hypotheses),
        "generation_seconds": generation_seconds,
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(
        VALIDATION_DIR / f"epoch_{epoch:02d}_validation.json",
        result,
    )
    write_json(
        VALIDATION_DIR / f"epoch_{epoch:02d}_predictions.json",
        {
            str(image_id): record
            for image_id, record in records.items()
        },
    )
    return result


def selection_key(result):
    return (
        float(result["metrics"]["CIDEr-D"]),
        float(result["metrics"]["BLEU-4"]),
        -float(result["teacher_forced_ce"]),
        -int(result["epoch"]),
    )


def append_step_log(row):
    with STEP_LOG.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )


def write_epoch_history(history):
    if not history:
        return
    rows = []
    for result in history:
        rows.append(
            {
                "epoch": result["epoch"],
                "train_combined_loss": result["train_combined_loss"],
                "train_ce": result["train_ce"],
                "train_kd_kl_t2": result["train_kd_kl_t2"],
                "validation_ce": result["teacher_forced_ce"],
                "BLEU-1": result["metrics"]["BLEU-1"],
                "BLEU-4": result["metrics"]["BLEU-4"],
                "METEOR": result["metrics"]["METEOR"],
                "ROUGE-L": result["metrics"]["ROUGE-L"],
                "CIDEr-D": result["metrics"]["CIDEr-D"],
                "generation_seconds": result["generation_seconds"],
            }
        )
    with EPOCH_HISTORY_CSV.open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        writer = csv.DictWriter(
            handle, fieldnames=list(rows[0])
        )
        writer.writeheader()
        writer.writerows(rows)


optimizer, scheduler, scaler = build_optimizer_state()
progress = {
    "epoch": 0,
    "cursor": 0,
    "global_optimizer_step": 0,
    "epoch_accumulators": {
        "combined_sum": 0.0,
        "ce_sum": 0.0,
        "kd_sum": 0.0,
        "valid_tokens": 0,
        "overflow_retries": 0,
    },
}
history = []
best_result = None
no_improve_epochs = 0
resume_source = None

if LAST_CHECKPOINT.is_file():
    resume_source = LAST_CHECKPOINT
elif INPUT_RESUME_CHECKPOINT is not None:
    resume_source = INPUT_RESUME_CHECKPOINT
    previous_root = resume_source.parent.parent
    for previous_directory, current_directory in [
        (previous_root / "logs", LOG_DIR),
        (previous_root / "validation", VALIDATION_DIR),
    ]:
        if previous_directory.is_dir():
            for previous_file in previous_directory.glob("*"):
                if previous_file.is_file():
                    current_file = (
                        current_directory / previous_file.name
                    )
                    if not current_file.exists():
                        shutil.copy2(previous_file, current_file)
    previous_best = resume_source.parent / BEST_CHECKPOINT.name
    if previous_best.is_file() and not BEST_CHECKPOINT.exists():
        shutil.copy2(previous_best, BEST_CHECKPOINT)

if resume_source is not None:
    progress, history, best_result, no_improve_epochs = (
        load_checkpoint(
            resume_source, optimizer, scheduler, scaler
        )
    )
    print("Resumed from              :", resume_source)
    print("Resume progress          :", progress)
else:
    print("Starting from locked Student initialization")

# Establish a recoverable optimizer-step boundary before entering the loop.
# On manual interruption, restore this last known-good checkpoint instead of
# serializing a potentially half-computed microbatch.
save_checkpoint(
    LAST_CHECKPOINT,
    optimizer, scheduler, scaler,
    dict(progress), history, best_result,
    no_improve_epochs,
)

TRAINING_COMPLETED = False
EARLY_STOPPED = False
INTERRUPTED_SAFELY = False
training_started_at = time.perf_counter()

try:
    while int(progress["epoch"]) < MAX_EPOCHS:
        epoch_index = int(progress["epoch"])
        epoch_number = epoch_index + 1
        cursor = int(progress["cursor"])
        loader = build_train_loader(epoch_index, cursor=cursor)
        iterator = iter(loader)
        accumulator = progress.get(
            "epoch_accumulators",
            {
                "combined_sum": 0.0,
                "ce_sum": 0.0,
                "kd_sum": 0.0,
                "valid_tokens": 0,
                "overflow_retries": 0,
            },
        )
        epoch_combined_sum = float(
            accumulator["combined_sum"]
        )
        epoch_ce_sum = float(accumulator["ce_sum"])
        epoch_kd_sum = float(accumulator["kd_sum"])
        epoch_valid_tokens = int(accumulator["valid_tokens"])
        epoch_images = cursor
        epoch_overflows = int(
            accumulator["overflow_retries"]
        )

        while True:
            microbatches = []
            for _ in range(ACCUMULATION):
                try:
                    microbatches.append(next(iterator))
                except StopIteration:
                    break
            if not microbatches:
                break
            result = optimizer_step(
                microbatches, optimizer, scheduler, scaler,
                retry_limit=10,
            )
            images = sum(
                len(batch["image_ids"]) for batch in microbatches
            )
            epoch_images += images
            progress["cursor"] = epoch_images
            progress["global_optimizer_step"] = (
                int(progress["global_optimizer_step"]) + 1
            )
            epoch_combined_sum += (
                result["combined_loss"] * result["valid_tokens"]
            )
            epoch_ce_sum += result["ce"] * result["valid_tokens"]
            epoch_kd_sum += (
                result["kd_kl_t2"] * result["valid_tokens"]
            )
            epoch_valid_tokens += result["valid_tokens"]
            epoch_overflows += result["overflow_retries"]
            progress["epoch_accumulators"] = {
                "combined_sum": epoch_combined_sum,
                "ce_sum": epoch_ce_sum,
                "kd_sum": epoch_kd_sum,
                "valid_tokens": epoch_valid_tokens,
                "overflow_retries": epoch_overflows,
            }
            step_row = {
                "created_at_utc": now_utc(),
                "epoch": epoch_number,
                "cursor": epoch_images,
                "global_optimizer_step": progress[
                    "global_optimizer_step"
                ],
                **result,
                "lr_backbone": optimizer.param_groups[0]["lr"],
                "lr_adapter": optimizer.param_groups[1]["lr"],
                "lr_decoder": optimizer.param_groups[2]["lr"],
            }
            append_step_log(step_row)
            if (
                int(progress["global_optimizer_step"]) % 50 == 0
            ):
                print(
                    f"epoch={epoch_number:02d} | "
                    f"step={progress['global_optimizer_step']:05d} | "
                    f"cursor={epoch_images:06d} | "
                    f"loss={result['combined_loss']:.6f}"
                )
            if (
                int(progress["global_optimizer_step"]) % 250 == 0
            ):
                save_checkpoint(
                    LAST_CHECKPOINT,
                    optimizer, scheduler, scaler,
                    dict(progress), history, best_result,
                    no_improve_epochs,
                )
        assert epoch_images == 113287
        assert epoch_valid_tokens > 0

        validation = evaluate_validation(epoch_number)
        validation.update(
            {
                "train_combined_loss": (
                    epoch_combined_sum / epoch_valid_tokens
                ),
                "train_ce": epoch_ce_sum / epoch_valid_tokens,
                "train_kd_kl_t2": epoch_kd_sum / epoch_valid_tokens,
                "train_valid_tokens": epoch_valid_tokens,
                "train_images": epoch_images,
                "overflow_retries": epoch_overflows,
                "global_optimizer_step": progress[
                    "global_optimizer_step"
                ],
            }
        )
        history.append(validation)

        previous_best_cider = (
            float(best_result["metrics"]["CIDEr-D"])
            if best_result is not None else float("-inf")
        )
        if (
            best_result is None
            or selection_key(validation) > selection_key(best_result)
        ):
            best_result = copy.deepcopy(validation)
            save_checkpoint(
                BEST_CHECKPOINT,
                optimizer, scheduler, scaler,
                {
                    "epoch": epoch_index + 1,
                    "cursor": 0,
                    "global_optimizer_step": progress[
                        "global_optimizer_step"
                    ],
                },
                history, best_result, no_improve_epochs,
            )

        if (
            float(validation["metrics"]["CIDEr-D"])
            >= previous_best_cider + 0.001
        ):
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1

        progress["epoch"] = epoch_index + 1
        progress["cursor"] = 0
        progress["epoch_accumulators"] = {
            "combined_sum": 0.0,
            "ce_sum": 0.0,
            "kd_sum": 0.0,
            "valid_tokens": 0,
            "overflow_retries": 0,
        }
        save_checkpoint(
            LAST_CHECKPOINT,
            optimizer, scheduler, scaler,
            dict(progress), history, best_result,
            no_improve_epochs,
        )
        write_epoch_history(history)
        print(
            f"epoch={epoch_number:02d} COMPLETE | "
            f"CIDEr-D={validation['metrics']['CIDEr-D']:.6f} | "
            f"best={best_result['metrics']['CIDEr-D']:.6f} | "
            f"CE={validation['teacher_forced_ce']:.6f}"
        )
        if (
            epoch_number >= 5
            and no_improve_epochs >= 4
        ):
            EARLY_STOPPED = True
            break

    TRAINING_COMPLETED = True
except KeyboardInterrupt:
    (
        progress,
        history,
        best_result,
        no_improve_epochs,
    ) = load_checkpoint(
        LAST_CHECKPOINT, optimizer, scheduler, scaler
    )
    INTERRUPTED_SAFELY = True
    print("Manual interruption captured.")
    print("Restored last verified optimizer-step boundary:", progress)

training_seconds_this_session = (
    time.perf_counter() - training_started_at
)
write_json(
    RUN_STATE_PATH,
    {
        "schema_version": "phase4r_run_state_v1",
        "created_at_utc": now_utc(),
        "status": (
            "complete" if TRAINING_COMPLETED
            else "interrupted_resume_ready"
        ),
        "progress": progress,
        "history_epochs": len(history),
        "best_result": best_result,
        "early_stopped": EARLY_STOPPED,
        "interrupted_safely": INTERRUPTED_SAFELY,
        "training_seconds_this_session": training_seconds_this_session,
        "last_checkpoint": str(LAST_CHECKPOINT),
        "last_checkpoint_sha256": (
            sha256_file(LAST_CHECKPOINT)
            if LAST_CHECKPOINT.is_file() else None
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04R CELL 5 STATUS")
print("=" * 72)
print(
    "Status                    :",
    "COMPLETE" if TRAINING_COMPLETED else "RESUME READY",
)
print("Completed epochs          :", len(history))
print("Global optimizer steps    :", progress["global_optimizer_step"])
print("Current cursor            :", progress["cursor"])
print("Best epoch                :", (
    best_result["epoch"] if best_result else None
))
print("Best CIDEr-D              :", (
    best_result["metrics"]["CIDEr-D"] if best_result else None
))
print("Early stopped             :", EARLY_STOPPED)
print("Interrupted safely        :", INTERRUPTED_SAFELY)
print("Last checkpoint           :", LAST_CHECKPOINT)
print("Test Set status           : SEALED")
print("Next                      : package complete or resume state")


Starting from locked Student initialization
epoch=01 | step=00050 | cursor=003200 | loss=7.508713
epoch=01 | step=00100 | cursor=006400 | loss=6.464878
epoch=01 | step=00150 | cursor=009600 | loss=5.493719
epoch=01 | step=00200 | cursor=012800 | loss=4.933237
epoch=01 | step=00250 | cursor=016000 | loss=4.587222
epoch=01 | step=00300 | cursor=019200 | loss=4.182466
epoch=01 | step=00350 | cursor=022400 | loss=3.910701
epoch=01 | step=00400 | cursor=025600 | loss=3.830682
epoch=01 | step=00450 | cursor=028800 | loss=3.657458
epoch=01 | step=00500 | cursor=032000 | loss=3.493164
epoch=01 | step=00550 | cursor=035200 | loss=3.481211
epoch=01 | step=00600 | cursor=038400 | loss=3.385050
epoch=01 | step=00650 | cursor=041600 | loss=3.392865
epoch=01 | step=00700 | cursor=044800 | loss=3.158438
epoch=01 | step=00750 | cursor=048000 | loss=3.102753
epoch=01 | step=00800 | cursor=051200 | loss=2.855021
epoch=01 | step=00850 | cursor=054400 | loss=2.948799
epoch=01 | step=00900 | cursor=057600 

PTBTokenizer tokenized 307821 tokens at 574496.27 tokens per second.
PTBTokenizer tokenized 50243 tokens at 251174.97 tokens per second.


{'testlen': 45244, 'reflen': 46029, 'guess': [45244, 40244, 35244, 30244], 'correct': [31149, 15012, 6439, 2717]}
ratio: 0.9829455343370271
epoch=01 COMPLETE | CIDEr-D=0.715426 | best=0.715426 | CE=2.927552
epoch=02 | step=01800 | cursor=001856 | loss=2.150760
epoch=02 | step=01850 | cursor=005056 | loss=1.988744
epoch=02 | step=01900 | cursor=008256 | loss=1.943719
epoch=02 | step=01950 | cursor=011456 | loss=1.911250
epoch=02 | step=02000 | cursor=014656 | loss=1.825864
epoch=02 | step=02050 | cursor=017856 | loss=1.889353
epoch=02 | step=02100 | cursor=021056 | loss=1.888850
epoch=02 | step=02150 | cursor=024256 | loss=1.832116
epoch=02 | step=02200 | cursor=027456 | loss=1.800313
epoch=02 | step=02250 | cursor=030656 | loss=1.823035
epoch=02 | step=02300 | cursor=033856 | loss=1.748620
epoch=02 | step=02350 | cursor=037056 | loss=1.844250
epoch=02 | step=02400 | cursor=040256 | loss=1.766450
epoch=02 | step=02450 | cursor=043456 | loss=1.661171
epoch=02 | step=02500 | cursor=046656

PTBTokenizer tokenized 307821 tokens at 694241.43 tokens per second.
PTBTokenizer tokenized 52241 tokens at 327111.98 tokens per second.


{'testlen': 47242, 'reflen': 47157, 'guess': [47242, 42242, 37242, 32242], 'correct': [33603, 17376, 7986, 3644]}
ratio: 1.0018024895561424
epoch=02 COMPLETE | CIDEr-D=0.887262 | best=0.887262 | CE=2.490019
epoch=03 | step=03550 | cursor=000512 | loss=1.681390
epoch=03 | step=03600 | cursor=003712 | loss=1.456904
epoch=03 | step=03650 | cursor=006912 | loss=1.642291
epoch=03 | step=03700 | cursor=010112 | loss=1.590792
epoch=03 | step=03750 | cursor=013312 | loss=1.493703
epoch=03 | step=03800 | cursor=016512 | loss=1.603896
epoch=03 | step=03850 | cursor=019712 | loss=1.603367
epoch=03 | step=03900 | cursor=022912 | loss=1.472397
epoch=03 | step=03950 | cursor=026112 | loss=1.535169
epoch=03 | step=04000 | cursor=029312 | loss=1.589548
epoch=03 | step=04050 | cursor=032512 | loss=1.656447
epoch=03 | step=04100 | cursor=035712 | loss=1.534998
epoch=03 | step=04150 | cursor=038912 | loss=1.473365
epoch=03 | step=04200 | cursor=042112 | loss=1.464696
epoch=03 | step=04250 | cursor=045312

PTBTokenizer tokenized 307821 tokens at 847625.03 tokens per second.
PTBTokenizer tokenized 52409 tokens at 303324.85 tokens per second.


... (31 outputs left out)

PTBTokenizer tokenized 307821 tokens at 678877.01 tokens per second.
PTBTokenizer tokenized 53542 tokens at 274216.45 tokens per second.


{'testlen': 48543, 'reflen': 48029, 'guess': [48543, 43543, 38543, 33543], 'correct': [36106, 19903, 10119, 5157]}
ratio: 1.0107018676216242
epoch=19 COMPLETE | CIDEr-D=1.091561 | best=1.091982 | CE=2.117577
epoch=20 | step=33650 | cursor=000064 | loss=1.041949
epoch=20 | step=33700 | cursor=003264 | loss=1.097109
epoch=20 | step=33750 | cursor=006464 | loss=1.059791
epoch=20 | step=33800 | cursor=009664 | loss=1.201038
epoch=20 | step=33850 | cursor=012864 | loss=1.116215
epoch=20 | step=33900 | cursor=016064 | loss=1.108694
epoch=20 | step=33950 | cursor=019264 | loss=1.165152
epoch=20 | step=34000 | cursor=022464 | loss=1.135512
epoch=20 | step=34050 | cursor=025664 | loss=1.254597
epoch=20 | step=34100 | cursor=028864 | loss=1.096423
epoch=20 | step=34150 | cursor=032064 | loss=1.237551
epoch=20 | step=34200 | cursor=035264 | loss=1.189027
epoch=20 | step=34250 | cursor=038464 | loss=1.191495
epoch=20 | step=34300 | cursor=041664 | loss=1.192040
epoch=20 | step=34350 | cursor=04486

PTBTokenizer tokenized 307821 tokens at 728901.18 tokens per second.
PTBTokenizer tokenized 53237 tokens at 312304.60 tokens per second.


{'testlen': 48238, 'reflen': 47820, 'guess': [48238, 43238, 38238, 33238], 'correct': [35903, 19773, 10013, 5100]}
ratio: 1.0087411125052068
epoch=20 COMPLETE | CIDEr-D=1.087135 | best=1.091982 | CE=2.117629
PHASE 04R CELL 5 STATUS
Status                    : COMPLETE
Completed epochs          : 20
Global optimizer steps    : 35420
Current cursor            : 0
Best epoch                : 16
Best CIDEr-D              : 1.09198154411797
Early stopped             : True
Interrupted safely        : False
Last checkpoint           : /kaggle/working//phase4_teacher_c_s3_logit_kd_seed2026/checkpoints/teacher_c_s3_logit_kd_seed2026_last.pt
Test Set status           : SEALED
Next                      : package complete or resume state


## Cell 6 — Package seed-2026 complete hoặc resumable state

- Complete run: ghi comparison với baseline S3 seed `2026`, manifest và
  marker sẵn sàng cho Phase 04S three-seed aggregate.
- Partial run: ghi resume marker chứa atomic last checkpoint. Attach output
  này rồi chạy lại đúng notebook seed `2026`.


In [6]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase4r_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4r_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4r_SHA256SUMS.txt"

if TRAINING_COMPLETED:
    assert best_result is not None
    assert BEST_CHECKPOINT.is_file()
    assert LAST_CHECKPOINT.is_file()
    kd_cider = float(best_result["metrics"]["CIDEr-D"])
    kd_bleu4 = float(best_result["metrics"]["BLEU-4"])
    kd_ce = float(best_result["teacher_forced_ce"])
    comparison = {
        "baseline_seed2026": {
            "CIDEr-D": BASELINE_CIDER,
            "BLEU-4": BASELINE_BLEU4,
            "validation_ce": BASELINE_CE,
        },
        "kd_seed2026": {
            "CIDEr-D": kd_cider,
            "BLEU-4": kd_bleu4,
            "validation_ce": kd_ce,
            "best_epoch": int(best_result["epoch"]),
        },
        "delta": {
            "CIDEr-D": kd_cider - BASELINE_CIDER,
            "BLEU-4": kd_bleu4 - BASELINE_BLEU4,
            "validation_ce": kd_ce - BASELINE_CE,
        },
        "kd_improves_cider_by_0_001": (
            kd_cider >= BASELINE_CIDER + 0.001
        ),
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    COMPARISON_PATH = RESULT_DIR / (
        "teacher_c_s3_logit_kd_seed2026_vs_baseline.json"
    )
    write_json(COMPARISON_PATH, comparison)
    NEXT_AUTHORIZATION = None
    FINAL_MARKER = OUTPUT_ROOT / (
        "PHASE4_TEACHER_C_S3_LOGIT_KD_SEED2026_FULL_RUN_COMPLETE.json"
    )
else:
    assert LAST_CHECKPOINT.is_file()
    NEXT_AUTHORIZATION = None
    comparison = None
    FINAL_MARKER = OUTPUT_ROOT / (
        "PHASE4_TEACHER_C_S3_LOGIT_KD_SEED2026_RESUME_READY.json"
    )

artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, FINAL_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4r_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

marker_payload = {
    "marker_version": (
        "phase4_teacher_c_s3_logit_kd_seed2026_"
        + ("full_run_complete_v1" if TRAINING_COMPLETED
           else "resume_ready_v1")
    ),
    "created_at_utc": now_utc(),
    "status": "complete" if TRAINING_COMPLETED else "resume_ready",
    "experiment_id": "P4C_C_S3_LOGIT",
    "seed": SEED,
    "progress": progress,
    "completed_epochs": len(history),
    "best_result": best_result,
    "last_checkpoint": {
        "relative_path": LAST_CHECKPOINT.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(LAST_CHECKPOINT),
    },
    "best_checkpoint": (
        {
            "relative_path": BEST_CHECKPOINT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(BEST_CHECKPOINT),
        }
        if BEST_CHECKPOINT.is_file() else None
    ),
    "comparison": comparison,
    "ready_for_phase04s_three_seed_aggregate": bool(TRAINING_COMPLETED),
    "manifest": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV),
        },
        "json": {
            "relative_path": MANIFEST_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_JSON),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHA256SUMS),
        },
    },
    "manifest_rows": len(manifest_rows),
    "manifest_verification": "all_passed",
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(FINAL_MARKER, marker_payload)

print("=" * 72)
print("PHASE 04R CELL 6 STATUS")
print("=" * 72)
print(
    "Status                    :",
    "COMPLETE" if TRAINING_COMPLETED else "RESUME READY",
)
print("Completed epochs          :", len(history))
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Last checkpoint           : VERIFIED")
print("Best checkpoint           :", BEST_CHECKPOINT.is_file())
print(
    "Phase 04S aggregate      :",
    "READY" if TRAINING_COMPLETED else "NOT YET",
)
print("Test Set status           : SEALED")
print("Marker                    :", FINAL_MARKER)
print(
    "Next                      :",
    (
        "After seed 123 and 2026 complete: Phase 04S aggregate"
        if TRAINING_COMPLETED
        else "Save Version, attach output and rerun seed 2026"
    ),
)


PHASE 04R CELL 6 STATUS
Status                    : COMPLETE
Completed epochs          : 20
Manifest rows             : 52
Manifest verification     : ALL PASSED
Last checkpoint           : VERIFIED
Best checkpoint           : True
Phase 04S aggregate      : READY
Test Set status           : SEALED
Marker                    : /kaggle/working//phase4_teacher_c_s3_logit_kd_seed2026/PHASE4_TEACHER_C_S3_LOGIT_KD_SEED2026_FULL_RUN_COMPLETE.json
Next                      : After seed 123 and 2026 complete: Phase 04S aggregate
